# Stage 3, each round: candidate briefings from the current student

The current student writes **four** briefings for every training packet at temperature 0.8. Back in the repo, `training/build_pairs_v3.py` scores every candidate with the reward (`training/reward.py`: hard rules, the simulated reader's learning, fit to the reader) and pairs each packet's best against its worst. No teacher text is involved: the student only ever moves toward its own best work.

Set `ADAPTER_NAME` to the adapter this round starts from and `ROUND` to the round number. Output: `MyDrive/news-briefer/candidates_r{ROUND}.jsonl`.

**Runtime → Change runtime type → L4 GPU**, then **Runtime → Run all**. Roughly 40 minutes on an L4 for 415 packets x 4; interrupted runs resume.


In [ ]:
import subprocess, sys, importlib

USE_UNSLOTH = True  # the fallback cell sets this to False

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip failed -- run the fallback cell below instead")

# unsloth pins its own tested versions of trl / peft / transformers / accelerate /
# bitsandbytes / datasets, so we pin unsloth to a release series and let it choose.
pip("--upgrade", "unsloth==2025.9.*", "unsloth_zoo==2025.9.*")
pip("matplotlib")

print()
print("installed versions:")
for name in ["unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "torch"]:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<14} {getattr(mod, '__version__', '?')}")
    except Exception as exc:  # noqa: BLE001
        print(f"  {name:<14} NOT IMPORTABLE ({exc})")

In [ ]:
import json, os, urllib.request
from google.colab import drive

ADAPTER_NAME = "news-briefer-dpo2-lora"   # the adapter this round starts from
ROUND = 1
CANDIDATES_PER_PACKET = 4
TEMPERATURE = 0.8

drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/news-briefer"
ADAPTER_DIR = os.path.join(WORK_DIR, ADAPTER_NAME)
assert os.path.isdir(ADAPTER_DIR), f"adapter not found at {ADAPTER_DIR}"
OUT = os.path.join(WORK_DIR, f"candidates_r{ROUND}.jsonl")

TRAIN_FILE = "sft_train.jsonl"
if not os.path.exists(TRAIN_FILE):
    urllib.request.urlretrieve(
        "https://github.com/sanjivp2703/adaptive-news-briefing-agent/releases/download/training-data-v7/" + TRAIN_FILE,
        TRAIN_FILE,
    )
with open(TRAIN_FILE, encoding="utf-8") as fh:
    train_rows = [json.loads(line) for line in fh if line.strip()]
print(len(train_rows), "training packets ->", OUT)


In [ ]:
import time, torch
from unsloth import FastLanguageModel

done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as fh:
        done = {json.loads(l)["id"] for l in fh if l.strip()}
print(len(done), "packets already generated; resuming")

model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=8192, dtype=None, load_in_4bit=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model)

t_start = time.time(); made = 0
with open(OUT, "a", encoding="utf-8") as out:
    for i, row in enumerate(train_rows, 1):
        rid = row["meta"]["id"]
        if rid in done:
            continue
        ids = tokenizer.apply_chat_template(row["messages"][:2], add_generation_prompt=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            gen = model.generate(input_ids=ids.repeat(CANDIDATES_PER_PACKET, 1), max_new_tokens=1200, temperature=TEMPERATURE,
                                 do_sample=True, top_p=0.95, pad_token_id=tokenizer.pad_token_id)
        candidates = [tokenizer.decode(g[ids.shape[1]:], skip_special_tokens=True) for g in gen]
        out.write(json.dumps({"id": rid, "profile": row["meta"].get("profile"), "round": ROUND, "adapter": ADAPTER_NAME,
                              "candidates": candidates}) + "\n")
        out.flush(); made += 1
        if made % 10 == 0:
            rate = (time.time() - t_start) / made
            print(f"  {i}/{len(train_rows)}  {rate:4.1f}s per packet  ~{rate * (len(train_rows) - i) / 60:.0f} min left")
print("wrote", OUT)


## Done

Download the file; the next step runs in the repo:

    PYTHONPATH=src:. .venv/bin/python training/build_pairs_v3.py ~/Downloads/candidates_r1.jsonl


In [ ]:
from google.colab import files
print(os.path.getsize(OUT) // 1024, "KB")
files.download(OUT)
